__IMPORTING REQUIRED LIBRARIES__

_Required libraries (fix order):_
1. polars (replacing pandas (novelty))
2. numpy
3. gensim
4. sentence-transformers
5. matplotlib
6. scikit-learn
7. os
8. dotenv
9. kagglehub
10. torch (for cuda)

In [2]:
import os
from dotenv import load_dotenv

import polars as pl
import numpy
import gensim
import torch

In [3]:
#wv = gensim.models.KeyedVectors.load_word2vec_format('word2vec_format/GoogleNews-vectors-negative300.bin', binary=True)

__LOADING DATASET__

_Dataset Details:_
1. Name - ISOT Fake News Dataset
2. Link - https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets/data
3. Number of rows - 20,826 (True) and 17,903 (False)

In [4]:
load_dotenv()

kaggle_key = os.getenv('KAGGLE_API_TOKEN')
kaggle_username = os.getenv('KAGGLE_USERNAME')

In [5]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [6]:
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets", output_dir='./dataset')

100%|██████████████████████████████████████████████████████████████████████████████| 41.0M/41.0M [00:09<00:00, 4.40MB/s]

Extracting files...


In [17]:
true_df = pl.read_csv('dataset/News _dataset/True.csv')
fake_df = pl.read_csv('dataset/News _dataset/Fake.csv')

__PROCESSING DATASET__

_We drop the date column from both datasets_
_We add a new column called label for both datasets, which adds a 1 label for true and -1 label for false (this will help in clssification later)_

___VERY IMPORTANT:___
_Using regular expression, we delete the mention of Reuters from true.csv as that will lead to model overfitting_

_We then merge both datasets to be later split for training and testing our RandomForest model (using cuML if possible)_

In [8]:
true_df.head()

title,text,subject,date
str,str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews""","""December 31, 2017 """
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews""","""December 29, 2017 """
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews""","""December 31, 2017 """
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews""","""December 30, 2017 """
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews""","""December 29, 2017 """


In [9]:
true_df = true_df.drop('date')

In [10]:
true_df.head()

title,text,subject
str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews"""
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews"""
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews"""
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews"""
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews"""


In [11]:
fake_df.head()

title,text,subject,date
str,str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News""","""December 31, 2017"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News""","""December 31, 2017"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News""","""December 30, 2017"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News""","""December 29, 2017"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News""","""December 25, 2017"""


In [12]:
fake_df = fake_df.drop("date")
fake_df.head()

title,text,subject
str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News"""


In [13]:
true_df = true_df.with_columns(pl.lit(1).alias("label"))
fake_df = fake_df.with_columns(pl.lit(-1).alias("label"))

# 3. Remove "Reuters" mentions (VERY IMPORTANT for leakage)
true_df = true_df.with_columns(
    pl.col("text").str.replace_all(r"(?i)reuters", "")
)

# Optional: also clean titles if needed
true_df = true_df.with_columns(
    pl.col("text").str.replace(
        r"^[A-Z\s\.,-]+\s*\(Reuters\)\s*-\s*",
        ""
    )
)
# 4. Merge datasets
df = pl.concat([true_df, fake_df])

# 5. Shuffle dataset (important before split)
df = df.sample(fraction=1.0, shuffle=True)

# 6. Train-test split
train_size = int(0.8 * df.height)

_We will use cuda if available to speed up SBERT embedding process_

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


__VECTORISING TEXT CONTENT__

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
# model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", device=device, trust_remote_code=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration.py: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

__TRAINING MODEL ON DATASET__

_We use RandomForest from cuML (or scikit-learn for fallback) to train our model to predict news classifications_ 

In [26]:
try:
    print("Importing cuML model")
    from cuml.ensemble import RandomForestClassifier
except Exception:
    print("Importing scikit-learn model")
    from sklearn.ensemble import RandomForestClassifier

Importing cuML model
